# Parkeringssystem - Den digitale parkeringsvakten

I forbindelsen med reduksjonen av parkeringsplasser på Gløshaugen har NTNU innført et avansert digitalt parkeringssystem. 
Systemet håndterer registrering av inn- og utkjøring, kapasitetskontroll, og anna surr. Ettersom jeg bare sykler så sender jeg oppgaven direkte videre til dere.

Husk at du kan bruke funksjoner fra tidligere oppgaver selv om du ikke har gjort dem, og at det finnes et par hjelpefunksjoner. Husk også at enkelte oppgaver er mer komplisert enn andre, så bruk tiden din fornuftig!

## Datastruktur

Systemet baserer seg på to hovedkomponenter:

**parking_lots** er en dictionary hvor nøklene er parkeringsplass-IDer og verdiene er en ny dictionary som inneholder informasjon om kapasitet, timepris og om det er ledig, for hver plass.

**transactions** er en liste som inneholder alle parkeringstransaksjoner - både aktive og fullførte. Hver transaksjon er en liste med [bilnummer, plass-ID, inn-tidspunkt, ut-tidspunkt]. Hvis ut-tidspunkt er None, betyr det at bilen fremdeles parkerer. Systemet bryr seg ikke om tid på døgnet, ikke datoer og slikt.

Du har også tilgang til to hjelpefunksjoner.

In [1]:
# [bil, plass, inn_tid, ut_tid] - betalt beregnes dynamisk
transactions = [
    ['AB123', 'petro', '06:00', '08:00'],    # Fullført parkering
    ['GH789', 'kjemi', '07:00', '09:00'],    # Fullført parkering
    ['IJ012', 'kjemi', '06:30', '08:45'],    # Fullført parkering
    ['KL345', 'petro', '08:00', '12:00'],     # Fullført parkering
    ['AB123', 'it', '09:30', None],          # Fremdeles parkert
    ['CD456', 'fys', '08:15', None],         # Fremdeles parkert 
    ['EF789', 'bygg', '10:00', None]        # Fremdeles parkert
]

parking_lots = {
    'kjemi': {'capacity': 20, 'price': 25, 'parked': 0},
    'it': {'capacity': 30, 'price': 25, 'parked': 1},  # AB123 står parkert, jamfør transactions
    'petro': {'capacity': 100, 'price': 15, 'parked': 0},
    'fys': {'capacity': 15, 'price': 15, 'parked': 1},  # ...
    'bygg': {'capacity': 1, 'price': 10, 'parked': 1}   # ...
}


# Hjelpefunksjoner
def calculate_hours(start_time, end_time):
    """Beregner antall påbegynte timer mellom to tidspunkt."""
    start_h, start_m = map(int, start_time.split(':'))
    end_h, end_m = map(int, end_time.split(':'))
    
    total_minutes = (end_h - start_h) * 60 + (end_m - start_m)
    hours = total_minutes // 60  
    if total_minutes % 60 > 0:   
        hours += 1
    
    return hours

# Funksjonen returnerer sortert en liste med navn på parkeringsplasser,
# sortert etter pris. Lavest pris først.
def sort_by_price(parking_spots):
    return sorted(parking_spots.keys(), key=lambda spot_id: parking_spots[spot_id]['price'])

## Oppgave 1: Kostnad (2%)

Skriv funksjonen `calculate_cost(start_time, end_time, price)`. Den skal returnere hvor mye det har kostet å parkere mellom disse tidspunktene.
- Kostnaden er *price* per påbegynte time.

Eksempel:
```python
>>> calculate_cost('08:00', '10:30', 25)
75  # 3 påbegynte timer × 25 kr = 75 kr
>>> calculate_cost('09:00', '09:45', 15)
15  # 1 påbegynt time × 15 kr = 15 kr
```

In [2]:
def calculate_cost(start_time, end_time, price):
    return calculate_hours(start_time, end_time) * price

print(f"Kostnad fra 08:00 til 10:30 @ 25kr/time (75): {calculate_cost('08:00', '10:30', 25)} kr")
print(f"Kostnad fra 09:00 til 09:45 @ 15kr/time (15): {calculate_cost('09:00', '09:45', 15)} kr")

Kostnad fra 08:00 til 10:30 @ 25kr/time (75): 75 kr
Kostnad fra 09:00 til 09:45 @ 15kr/time (15): 15 kr


## Oppgave 2: Finn aktiv parkering for bil (3%)

Systemet må kunne finne hvor en bil parkerer for øyeblikket. Lag funksjonen `find_active_parking(transactions, plate_number)` som:

- har to parametre
    - *transactions*: alle parkeringsregistreringene.
    - *plate_number*: registreringsnummeret
- returnerer en liste *[lot, start_time]* hvis bilen står på parkert på *lot* nå.
- returnerer *None* hvis bilen ikke har noen aktiv parkering.

Eksempel:
```python
>>> find_active_parking(transactions, 'AB123')
['it', '09:30']
>>> find_active_parking(transactions, 'GH789')
None
```

In [3]:
def find_active_parking(transactions, plate_number):
    for s, p, i, o in transactions:
        if s == plate_number and not o:
            return [p, i]

print(f"AB123 parkerer: {find_active_parking(transactions, 'AB123')}")
print(f"GH789 parkerer: {find_active_parking(transactions, 'GH789')}")


AB123 parkerer: ['it', '09:30']
GH789 parkerer: None


## Oppgave 3: Hent data fra parkeringsplass (3%)

Systemet trenger en generisk funksjon for å hente informasjon om en spesifikk parkeringsplass. Lag funksjonen `get_data_from_lot(parking_lots, lot, tag)` som:

- Har tre parametre
    - *parking_lots*: informasjon om alle parkeringsplassene
    - *lot*: navnet på parkeringsplassen
    - *tag*: navnet på datafeltet som skal hentes (f.eks. 'parked', 'capacity', 'price')
- returnerer verdien av det spesifikke datafeltet
- returnerer 0 hvis lot ikke eksisterer eller tag ikke finnes

Eksempel:
```python
>>> get_data_from_lot(parking_lots, 'kjemi', 'parked')
0
>>> get_data_from_lot(parking_lots, 'it', 'parked') 
1
>>> get_data_from_lot(parking_lots, 'kjemi', 'capacity')
20
```

In [4]:
def get_data_from_lot(parking_lots, lot, tag):
    return parking_lots.get(lot, {}).get(tag, 0)

print(f"Kapasitet på kjemi: {get_data_from_lot(parking_lots, 'kjemi', 'parked')}")
print(f"Antall biler parkert på it: {get_data_from_lot(parking_lots, 'it', 'parked')}")

Kapasitet på kjemi: 0
Antall biler parkert på it: 1


## Oppgave 4: Sjekk om det er ledig plass (2%)

Før nye biler kan parkere, må systemet sjekke om det er ledig kapasitet. Lag funksjonen `has_available_space(parking_lots, lot)`. 

Funksjonen har to parametre: 
    - *parking_lots*: informasjon om alle parkeringsplassene
    - *lot*: navnet på parkeringsplassen.
- Bruker `get_data_from_lot` fra oppgave 3 for å hente 'parked' og 'capacity'
- returnerer *True* hvis det er ledig på denne parkeringsplassen, *False* hvis ikke.
- returnerer *False* hvis lot ikke eksisterer

Eksempel:
```python
>>> has_available_space(parking_lots, 'kjemi')  
True   # 20 plasser, ingen parkert
>>> has_available_space(parking_lots, 'bygg')
False # 1 plass, 1 parkert
```

In [5]:
def has_available_space(parking_lots, lot):
    return get_data_from_lot(parking_lots, lot, 'parked') < get_data_from_lot(parking_lots, lot, 'capacity')
     
print(f"kjemi har ledig plass, for den gir: {has_available_space(parking_lots, 'kjemi')}")
print(f"bygg har ikke ledig plass, for den gir: {has_available_space(parking_lots, 'bygg')}")

kjemi har ledig plass, for den gir: True
bygg har ikke ledig plass, for den gir: False


## Oppgave 5: Registrer ny parkering (5%)

Når en ny bil kommer til parkeringsanlegget, må systemet tildele en ledig parkeringsplass. Hvis det er en! Lag funksjonen `register_parking(parking_lots, transactions, plate_number, time)` som:

- Returnerer navnet på en parkeringsplass som har en ledig plass
- Alltid forsøker å henvise til den parkeringsplassen med lavest avgift
- Bruker *time* som inn-tidspunkt for transaksjonen
- Returnerer *lot* for den tildelte plassen hvis vellykket  
- Utløser *ValueError* hvis alle plasser er fulle
- Utløser *ValueError* hvis bilen allerede parkerer et sted (tomt unntak ok)
- Oppdaterer verdier der det er nødvendig for å opprettholde korrekt tilstand

Eksempel:
```python
>>> register_parking(parking_lots, transactions, 'NY123', '12:00')
'bygg'  # Billigste ledige plass (10 kr) - men den har kun 1 plass
>>> register_parking(parking_lots, transactions, 'AB123', '12:00')  # Parkerer allerede
ValueError
```

In [6]:
def register_parking(parking_lots, transactions, plate_number, time):
    
    if find_active_parking(transactions, plate_number):
        raise ValueError("Parkert allerede")
    
    for lot in sort_by_price(parking_lots):
        if has_available_space(parking_lots, lot):
            parking_lots[lot]['parked'] += 1
            transactions.append([plate_number, lot, time, None])
            return lot
    
    raise ValueError("Fullt overalt")

plass1 = register_parking(parking_lots, transactions, 'NY123', '12:00')
print(f"NY123 fikk plass: {plass1}")

try:
    plass2 = register_parking(parking_lots, transactions, 'AB123', '12:00')
    print(f"AB123 fikk plass: {plass2}")
except ValueError as e:
    print(f"Feil: {e}")

NY123 fikk plass: petro
Feil: Parkert allerede


## Oppgave 6: Registrer utkjøring og beregn kostnad (4%)

Når en bil forlater parkeringsanlegget, må systemet registrere utkjøringen og beregne kostnaden. Lag funksjonen `checkout_car(parking_lots, transactions, plate_number, end_time)` som:

- Returnerer kostnaden for parkeringen for bilen med dette registreringsnummeret.
- Returnerer 0 hvis bilen ikke har en aktiv parkering.
- Kostnad er basert på *antall påbegynte timer og timepris*, og returneres som heltall
- Oppdaterer verdier der det er nødvendig for å opprettholde korrekt tilstand
- Returnerer totalkostnaden som heltall

**Hint:** Bruk hjelpefunksjonen `calculate_hours()` for å beregne antall timer.

Eksempel:
```python
>>> checkout_car(parking_lots, transactions, 'AB123', '12:30')
75  # 3 timer × 25 kr/time = 75 kr
>>> checkout_car(parking_lots, transactions, 'XY999', '12:30')
0
```

In [7]:
def checkout_car(parking_lots, transactions, plate_number, end_time):
    data = find_active_parking(transactions, plate_number)
    if not data: return 0 
    lot, start_time = data # Vi må teste før vi kan pakke ut, kan _ikke_ kjøre l, s = find_active_parking(...)
    parking_lots[lot]['parked'] -= 1
    
    for t in transactions:
        if t[0] == plate_number and t[3] == None:
            t[3] = end_time
            
    price = get_data_from_lot(parking_lots, lot, "price")
    return calculate_cost(start_time, end_time, price)

kostnad = checkout_car(parking_lots, transactions, 'AB123', '12:30')
print(f"Kostnad for AB123 (75 kr): {kostnad} kr")

print("Er AB123 aktiv (None):",find_active_parking(transactions, 'AB123'))

kostnad2 = checkout_car(parking_lots, transactions, 'XY999', '12:30')
print(f"Kostnad for XY999 som ikke har parkert (0): {kostnad2}")

Kostnad for AB123 (75 kr): 75 kr
Er AB123 aktiv (None): None
Kostnad for XY999 som ikke har parkert (0): 0


## Oppgave 7: Hent parkeringsinfo for bil (5%)

For kundeservice og rapportering trenger systemet å kunne vise detaljert informasjon om alle fullførte parkeringer for ett registreringsnummer. Lag funksjonen `get_parking_info(parking_lots, transactions, plate_number)` som:

- Funksjonen skal returnere en liste med informasjon om hver fullførte parkering
- Hvert felt i listen er en dictionary, som inneholder følgende om hver parkering:
    - 'lot', 'start', 'end', 'cost'
- Funksjonen skal returnere en tom liste hvis registreringsnummeret ikke har gjennomført en parkering ennå.

Eksempel:
```python
>>> get_parking_info(parking_lots, transactions, 'AB123')
[{'lot': 'it', 'start': '09:30', 'end': '12:30', 'cost': 75}, {'lot': 'petro', 'start': '06:00', 'end': '08:00', 'cost': 30}]
>>> get_parking_info(parking_lots, transactions, 'XY999')
[]  # Hvis XY999 ikke har fullført parkering ennå
```

In [8]:
def get_parking_info(parking_lots, transactions, plate_number):
    out = []
    for p, l, s, e in transactions:
        if p == plate_number and e:
            cost = calculate_cost(s, e, get_data_from_lot(parking_lots, l, "price"))
            out.append({'lot':l, 'start':s, 'end':e, 'cost':cost})

    return out

info = get_parking_info(parking_lots, transactions, 'AB123')
print(f"Parkeringsinfo for AB123: {info}")

info2 = get_parking_info(parking_lots, transactions, 'XY999')
print(f"Parkeringsinfo for XY999: {info2}")

Parkeringsinfo for AB123: [{'lot': 'petro', 'start': '06:00', 'end': '08:00', 'cost': 30}, {'lot': 'it', 'start': '09:30', 'end': '12:30', 'cost': 75}]
Parkeringsinfo for XY999: []


## Oppgave 8: Finn mest lojale kunde (6%)

For markedsføring og kundeanalyse trenger systemet å finne den mest lojale kunden. Lag funksjonen `find_most_loyal_customer(parking_lots, transactions)` som:

- Finner kunden som har parkert flest ganger (fullførte parkeringer)
- Hvis flere kunder har likt antall besøk, velges den med høyest total kostnad
- Returnerer et tuppel: `(bilnummer, antall_besøk, total_kostnad)`
- Returnerer `None` hvis ingen har fullført parkeringer

**Hint:** Bruk `get_parking_info()` fra oppgave 7 for å hente data per bil.

Eksempel med initial data:
```python
>>> find_most_loyal_customer(parking_lots, transactions)
('AB123', 1, 30)  # AB123 har 1 besøk med totalt 30kr
```

In [ ]:
def find_most_loyal_customer(parking_lots, transactions):
    # Finn alle unike biler med fullførte parkeringer
    completed_plates = set()
    for plate, _, _, end in transactions:
        if end:
            completed_plates.add(plate)
    
    if not completed_plates:
        return None
    
    best_plate = None
    best_visits = 0
    best_total = 0
    
    for plate in completed_plates:
        info = get_parking_info(parking_lots, transactions, plate)
        if info:
            visits = len(info)
            total = sum(parking['cost'] for parking in info)
            print(f"total for {plate}: {total}")
            
            if visits > best_visits or (visits == best_visits and total > best_total):
                best_plate = plate
                best_visits = visits
                best_total = total
    
    return (best_plate, best_visits, best_total)

# Lager testdata, og oppsummerer status etter hvert felt.
test_transactions = [
    ['AB123', 'petro', '06:00', '08:00'],    # AB123: 30kr
    ['AB123', 'kjemi', '13:00', '15:30'],    # AB123: 75kr  
    ['AB123', 'fys', '16:00', '17:00'],      # AB123: 15kr (totalt 3 besøk, 120kr)
    ['GH789', 'kjemi', '07:00', '09:00'],    # GH789: 50kr
    ['GH789', 'petro', '10:00', '14:00'],    # GH789: 60kr (totalt 2 besøk, 110kr)
    ['IJ012', 'kjemi', '06:30', '08:45'],    # IJ012: 75r
    ['IJ012', 'it', '09:00', '11:30'],       # IJ012: 75kr (totalt 2 besøk, 150kr)
    ['KL345', 'petro', '08:00', '12:00'],    # KL345: 60kr (totalt 1 besøk)
    ['NY123', 'kjemi', '14:00', '16:00'],    # NY123: 50kr (totalt 1 besøk)
    ['CD456', 'fys', '08:15', None],         # Aktiv - skal ikke telles
    ['EF789', 'bygg', '10:00', None]         # Aktiv - skal ikke telles
]

loyal = find_most_loyal_customer(parking_lots, test_transactions)
print(f"Mest lojale kunde: {loyal}")
print(f"Forventet: ('AB123', 3, 120)")
print(f"\nForklaring:")
print(f"  AB123: 3 besøk (30 + 75 + 15 = 120kr)")
print(f"  IJ012: 2 besøk (56 + 75 = 131kr)")  
print(f"  GH789: 2 besøk (50 + 60 = 110kr)")
print(f"  AB123 vinner pga flest besøk (3 > 2)")


total for AB123: 120
total for GH789: 110
total for IJ012: 150
total for KL345: 60
total for NY123: 50
Mest lojale kunde: ('AB123', 3, 120)
Forventet: ('AB123', 3, 120)

Forklaring:
  AB123: 3 besøk (30 + 75 + 15 = 120kr)
  IJ012: 2 besøk (56 + 75 = 131kr)
  GH789: 2 besøk (50 + 60 = 110kr)
  AB123 vinner pga flest besøk (3 > 2)


# Ikke en eksamensoppgave
Til slutt legger jeg ved et atskillig større eksempel på bruk av funksjonene! Som ble mekket ved hjelp av Claude 4.5, må jeg innrømme.

In [10]:
# Omfattende demonstrasjon av parkeringssystemet
# Simulerer en hel dag med parkering på NTNU Gløshaugen

# Reset til initial tilstand
parking_lots = {
    'kjemi': {'capacity': 20, 'price': 25, 'parked': 0},
    'it': {'capacity': 30, 'price': 25, 'parked': 0},
    'petro': {'capacity': 100, 'price': 15, 'parked': 0},
    'fys': {'capacity': 15, 'price': 15, 'parked': 0},
    'bygg': {'capacity': 1, 'price': 10, 'parked': 0}
}

transactions = []

print("=" * 80)
print("PARKERINGSSYSTEM - DAGLIG SIMULERING")
print("=" * 80)

# ============================================================================
# MORGENEN (06:00 - 09:00) - Mange ansatte ankommer
# ============================================================================
print("\n📅 MORGEN (06:00 - 09:00)")
print("-" * 80)

morning_arrivals = [
    ('AA001', '06:00'), ('AA002', '06:15'), ('AA003', '06:30'),
    ('BB001', '07:00'), ('BB002', '07:15'), ('BB003', '07:30'),
    ('CC001', '08:00'), ('CC002', '08:15'), ('CC003', '08:30'),
    ('DD001', '08:45'), ('DD002', '08:50'), ('DD003', '08:55')
]

for plate, time in morning_arrivals:
    try:
        lot = register_parking(parking_lots, transactions, plate, time)
        print(f"✓ {time}: {plate} parkert på {lot} ({parking_lots[lot]['price']} kr/time)")
    except ValueError as e:
        print(f"✗ {time}: {plate} - Feil: {e}")

# ============================================================================
# FORMIDDAG (09:00 - 12:00) - Noen reiser, flere ankommer
# ============================================================================
print("\n📅 FORMIDDAG (09:00 - 12:00)")
print("-" * 80)

# Noen tidlige reiser hjem
early_departures = [
    ('AA001', '09:00'), ('AA002', '09:30'), ('BB001', '10:00')
]

for plate, time in early_departures:
    cost = checkout_car(parking_lots, transactions, plate, time)
    print(f"💰 {time}: {plate} sjekket ut - Betalt: {cost} kr")

# Flere ankommer
forenoon_arrivals = [
    ('EE001', '09:00'), ('EE002', '09:30'), ('EE003', '10:00'),
    ('FF001', '10:30'), ('FF002', '11:00'), ('FF003', '11:30')
]

for plate, time in forenoon_arrivals:
    try:
        lot = register_parking(parking_lots, transactions, plate, time)
        print(f"✓ {time}: {plate} parkert på {lot}")
    except ValueError as e:
        print(f"✗ {time}: {plate} - {e}")

# ============================================================================
# LUNSJ (12:00 - 13:00) - Noen kjører ut for lunsj
# ============================================================================
print("\n📅 LUNSJTID (12:00 - 13:00)")
print("-" * 80)

lunch_departures = [
    ('CC001', '12:00'), ('CC002', '12:15'), ('DD001', '12:30')
]

for plate, time in lunch_departures:
    cost = checkout_car(parking_lots, transactions, plate, time)
    print(f"💰 {time}: {plate} drar til lunsj - Betalt: {cost} kr")

# Noen returnerer fra lunsj og prøver å parkere igjen
lunch_returns = [
    ('CC001', '13:00'), ('CC002', '13:15')
]

for plate, time in lunch_returns:
    try:
        lot = register_parking(parking_lots, transactions, plate, time)
        print(f"✓ {time}: {plate} tilbake fra lunsj, parkert på {lot}")
    except ValueError as e:
        print(f"✗ {time}: {plate} - {e}")

# ============================================================================
# ETTERMIDDAG (13:00 - 16:00) - Flere møter, noen reiser tidlig
# ============================================================================
print("\n📅 ETTERMIDDAG (13:00 - 16:00)")
print("-" * 80)

afternoon_arrivals = [
    ('GG001', '13:00'), ('GG002', '13:30'), ('HH001', '14:00'),
    ('HH002', '14:30'), ('II001', '15:00'), ('II002', '15:30')
]

for plate, time in afternoon_arrivals:
    try:
        lot = register_parking(parking_lots, transactions, plate, time)
        print(f"✓ {time}: {plate} parkert på {lot}")
    except ValueError as e:
        print(f"✗ {time}: {plate} - {e}")

# Noen reiser hjem tidlig
early_home = [
    ('AA003', '14:00'), ('BB002', '14:30'), ('BB003', '15:00')
]

for plate, time in early_home:
    cost = checkout_car(parking_lots, transactions, plate, time)
    print(f"💰 {time}: {plate} reiser hjem - Betalt: {cost} kr")

# ============================================================================
# KVELD (16:00 - 18:00) - Arbeidsdagens slutt
# ============================================================================
print("\n📅 KVELD (16:00 - 18:00)")
print("-" * 80)

evening_departures = [
    ('CC003', '16:00'), ('DD002', '16:15'), ('DD003', '16:30'),
    ('EE001', '16:45'), ('EE002', '17:00'), ('EE003', '17:15'),
    ('FF001', '17:30'), ('FF002', '17:45'), ('FF003', '18:00'),
    ('GG001', '18:00'), ('GG002', '18:00'), ('HH001', '18:00'),
    ('HH002', '18:00'), ('II001', '18:00'), ('II002', '18:00'),
    ('CC001', '18:00'), ('CC002', '18:00')
]

for plate, time in evening_departures:
    cost = checkout_car(parking_lots, transactions, plate, time)
    if cost > 0:
        print(f"💰 {time}: {plate} reiser hjem - Betalt: {cost} kr")

# ============================================================================
# FEILHÅNDTERING - Demonstrer edge cases
# ============================================================================
print("\n⚠️  FEILHÅNDTERING")
print("-" * 80)

# Forsøk å parkere samme bil to ganger
print("\n1. Dobbel-registrering:")
register_parking(parking_lots, transactions, 'TEST001', '18:00')
print(f"   ✓ TEST001 parkert første gang")
try:
    register_parking(parking_lots, transactions, 'TEST001', '18:15')
except ValueError as e:
    print(f"   ✗ Kan ikke parkere TEST001 igjen: {e}")

# Forsøk å sjekke ut bil som ikke parkerer
print("\n2. Checkout av ikke-parkert bil:")
cost = checkout_car(parking_lots, transactions, 'FINNES_IKKE', '18:00')
print(f"   Kostnad for ikke-eksisterende bil: {cost} kr (forventet 0)")

# ============================================================================
# SLUTTRAPPORT
# ============================================================================
print("\n" + "=" * 80)
print("DAGLIG RAPPORT")
print("=" * 80)

print("\n📊 STATUS PER PARKERINGSPLASS:")
print("-" * 80)
for lot, data in parking_lots.items():
    print(f"  {lot:8s}: {data['parked']:2d}/{data['capacity']:3d} plasser opptatt "
          f"(pris: {data['price']} kr/time)")

print("\n👥 KUNDEANALYSE:")
print("-" * 80)
loyal = find_most_loyal_customer(parking_lots, transactions)
if loyal:
    plate, visits, total = loyal
    print(f"  Mest lojale kunde: {plate}")
    print(f"  Antall besøk: {visits}")
    print(f"  Total kostnad: {total} kr")
    
    # Vis detaljer for mest lojale kunde
    info = get_parking_info(parking_lots, transactions, plate)
    print(f"\n  Detaljert historikk for {plate}:")
    for i, parking in enumerate(info, 1):
        print(f"    {i}. {parking['lot']:8s} {parking['start']} - {parking['end']} "
              f"({parking['cost']} kr)")

print("\n💵 DAGLIG STATISTIKK:")
print("-" * 80)
total_revenue = sum(
    calculate_cost(start, end, get_data_from_lot(parking_lots, lot, 'price'))
    for _, lot, start, end in transactions if end
)
completed_parkings = sum(1 for _, _, _, end in transactions if end)
print(f"  Totale inntekter: {total_revenue} kr")
print(f"  Fullførte parkeringer: {completed_parkings}")
print(f"  Gjennomsnittlig kostnad: {total_revenue // completed_parkings if completed_parkings > 0 else 0} kr")

print("\n🚗 FORTSATT PARKERT:")
print("-" * 80)
still_parked = [(plate, lot, start) for plate, lot, start, end in transactions if not end]
if still_parked:
    for plate, lot, start in still_parked:
        print(f"  {plate} på {lot} siden {start}")
else:
    print("  Ingen biler står parkert")

print("\n" + "=" * 80)
print("SIMULERING FULLFØRT")
print("=" * 80)

PARKERINGSSYSTEM - DAGLIG SIMULERING

📅 MORGEN (06:00 - 09:00)
--------------------------------------------------------------------------------
✓ 06:00: AA001 parkert på bygg (10 kr/time)
✓ 06:15: AA002 parkert på petro (15 kr/time)
✓ 06:30: AA003 parkert på petro (15 kr/time)
✓ 07:00: BB001 parkert på petro (15 kr/time)
✓ 07:15: BB002 parkert på petro (15 kr/time)
✓ 07:30: BB003 parkert på petro (15 kr/time)
✓ 08:00: CC001 parkert på petro (15 kr/time)
✓ 08:15: CC002 parkert på petro (15 kr/time)
✓ 08:30: CC003 parkert på petro (15 kr/time)
✓ 08:45: DD001 parkert på petro (15 kr/time)
✓ 08:50: DD002 parkert på petro (15 kr/time)
✓ 08:55: DD003 parkert på petro (15 kr/time)

📅 FORMIDDAG (09:00 - 12:00)
--------------------------------------------------------------------------------
💰 09:00: AA001 sjekket ut - Betalt: 30 kr
💰 09:30: AA002 sjekket ut - Betalt: 60 kr
💰 10:00: BB001 sjekket ut - Betalt: 45 kr
✓ 09:00: EE001 parkert på bygg
✓ 09:30: EE002 parkert på petro
✓ 10:00: EE003 par